# Création du fichier d'export des résultats

Le fichier ne contient que les DPE a mettre à jour (les résultats identiques au rappro de 2024 sont exclus)

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [18]:
# Export pour MAJ de la base DPE

df_export = pd.read_csv('notebooks/rapprochements/DPE/2026/results_DPE_RNB.csv')

In [19]:
df_export.sample(20)

,numero_dpe,id_rnb,provenance_id_rnb,file,rnb_id_rappro
13201354,2557E2699395G,NaN,NaN,existant,['2RFCJW2EYGQH']
14447632,2406E4301985I,NaN,NaN,existant,"['FK5MNZQ4TCJ6', 'ZNVWC33MSYV4', 'A1CTK7KY13XZ..."
16374046,2331E3803588D,7RW8ECTMXRSJ,Reprise RNB,existant,['7RW8ECTMXRSJ']
16748816,2630E1661993E,NGA4SD2DX4TM,Logiciel,existant,['NGA4SD2DX4TM']
10302486,2459E2056622G,H946K3GAB4SG,Reprise RNB,existant,['H946K3GAB4SG']
8741401,2360E0418500O,CYRN6ZRARNVH,Reprise RNB,existant,['CYRN6ZRARNVH']
9769784,2476E1958271E,NaN,NaN,existant,"['PJQ2JTQNP4FK', 'MF1DJM6VNZDK']"
16972324,2603E1777884M,NaN,NaN,existant,['S6GHXH2TVB56']
2801201,2151E0232302R,EX3TDDGAYH1W,Reprise RNB,existant,['EX3TDDGAYH1W']
6260969,2377E0486973U,3G7897SC64XB,Reprise RNB,existant,['3G7897SC64XB']


### Selection des lignes pour export

In [20]:

# tous les dpe dont la colonne id_rnb est actuellement vide
df_export_new_dpe = df_export[(df_export['id_rnb'].isna()) & (df_export['rnb_id_rappro'].apply(len) == 16)]
df_export_new_dpe['rnb_id_rappro'] = df_export_new_dpe['rnb_id_rappro'].apply(lambda id: id.strip("'[]"))

# les dpe actuellement en "reprise RNB"
df_export_reprise = df_export[(df_export['provenance_id_rnb'] == 'Reprise RNB')]

# les dpe actuellement en "reprise RNB", mais qui lors du rapprochement ne sont plus rapprochés
# - car pas de résultat trouvé
# - car trop de résultats trouvés
# 16 caractères car ['FZG8NZVQ3QQ8'] => 12 caractères d'ID et 4 de ['']
df_export_reprise_no_result = df_export_reprise[df_export_reprise['rnb_id_rappro'].apply(len) != 16]
# on va mettre à jour côté DPE qu'on supprime l'ancien ID-RNB
df_export_reprise_no_result['rnb_id_rappro'] = 'delete_existing_result'

# les dpe actuellement en "reprise RNB", qui sont rapprochés avec succès
df_export_reprise_success = df_export_reprise[(df_export_reprise['rnb_id_rappro'].apply(len) == 16)]
df_export_reprise_success['rnb_id_rappro'] = df_export_reprise_success['rnb_id_rappro'].apply(lambda id: id.strip("'[]"))
df_export_reprise_success = df_export_reprise_success[df_export_reprise_success['rnb_id_rappro'] != df_export_reprise_success['id_rnb']]

df_export_final = pd.concat([df_export_new_dpe, df_export_reprise_no_result, df_export_reprise_success])

In [30]:
# nouveau ID à renseigner
df_export_new_dpe.shape

(1566963, 5)

In [ ]:
# résultat à supprimer
df_export_reprise_no_result.shape

(205498, 5)

In [ ]:
# à mettre à jour
df_export_reprise_success.shape

(74229, 5)

In [33]:
df_export_final.sample(20)

,numero_dpe,id_rnb,provenance_id_rnb,file,rnb_id_rappro
16827200,2659E1622151X,NaN,NaN,existant,9YG17WK53ADY
8563129,2506E3115893L,NaN,NaN,existant,BGQA73C5X3YD
9166242,2409E1888044O,1MZMB9PT8MMX,Reprise RNB,existant,delete_existing_result
13920580,2535E2348555O,NaN,NaN,existant,7MYCY9KK51EZ
17236695,2670E1978983Q,NaN,NaN,existant,KY54B4FH1W4Y
10816405,2459E2953301X,CYDWDMHP2W57,Reprise RNB,existant,delete_existing_result
1974009,2634E0029301M,NaN,NaN,existant,RPNKMTEWR6S1
10354247,2477E1787811B,W3Q42NQGMX2Y,Reprise RNB,existant,delete_existing_result
16301694,2627E1349303N,NaN,NaN,existant,SRTBQSJ69KZQ
14884716,2531E2621827W,NaN,NaN,existant,TCFKRPTBPYG8


In [35]:
# export du fichier
df_export_final[['numero_dpe', 'file', 'rnb_id_rappro']].to_csv('notebooks/rapprochements/DPE/2026/update_rapprochement_DPE_RNB.csv', index=False)